# Fit the power law, then spend the budget

**Topic:** Scaling laws &nbsp;·&nbsp; **Difficulty:** Medium

Loss falls as a power law in parameters and in data, with an irreducible floor:
`L = E + A/N^alpha + B/D^beta`. Once you have fitted those constants you can do
the thing the constants are for: given a compute budget, decide how big a model
to train and for how many tokens.

Fit the law to a grid of runs, then answer the allocation question.

## The data (created in the next cell)

- `runs`: 40 observed `(N, D, loss)` triples, generated from a known law plus
  small noise.
- Compute is `C = 6 N D`.
- `budgets = [1e19, 1e20, 1e21]` FLOPs.

## Your task

With **NumPy only**:

1. Fit `E, A, alpha, B, beta` directly, not by fitting `log(loss - E)`,
   which needs `E` first: grid-search `alpha` and `beta` over
   `np.linspace(0.2, 0.8, 25)` each. For each pair solve for `E, A, B` by linear
   least squares on the features `[1, N^-alpha, D^-beta]`, and keep the pair with
   the lowest residual.
2. For each budget, search `N` over `np.logspace(7, 11, 400)`, set
   `D = C / (6 N)`, evaluate the fitted loss, and record the minimizing `N`, the
   corresponding `D`, and the ratio `D/N`.
3. Set `answers` to a dict with `"alpha"`, `"beta"`, `"E"`, `"opt_N"` (3 values),
   `"opt_D"` (3 values), `"tokens_per_param"` (3 values).

## Evaluation

Compared to the reference to a relative tolerance of 1e-5, with `alpha`, `beta`
and `E` as plain scalars, not one-element arrays. The evaluation cell
checks the recovered exponents are close to the ones used to generate the data
and that the optimal tokens-per-parameter ratio is roughly constant across
budgets.

In [ ]:
import numpy as np

rng = np.random.default_rng(4006)
E_true, A_true, al_true = 1.65, 4.2e3, 0.34
B_true, be_true = 2.1e4, 0.29

Ns = np.logspace(7.5, 10.0, 8)
Ds = np.logspace(9.0, 11.5, 5)
runs = []
for N in Ns:
    for D in Ds:
        L = E_true + A_true / N ** al_true + B_true / D ** be_true
        runs.append((N, D, L * (1 + rng.normal(0, 0.002))))
runs = np.array(runs)
budgets = [1e19, 1e20, 1e21]
print(f"{len(runs)} runs")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("alpha", [], 0.32500000000000007, -0.054417354814712236, 1e-05),
    ("beta", [], 0.30000000000000004, -0.05023140444434975, 1e-05),
    ("E", [], 2.2164358436100775, -0.3711156176177711, 1e-05),
    ("opt_N", [3], 312288485.5613742, -316859165.416665, 1e-05),
    ("opt_D", [3], 1071849247880.0781, -1141089250036.637, 1e-05),
    ("tokens_per_param", [3], 9618.78538808504, -5329.388547393159, 1e-05),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

if "tokens_per_param" in answers:
    _t = np.asarray(answers["tokens_per_param"], dtype=float)
    _stable = float(_t.max() / _t.min()) < 1.5
    _exp = abs(answers["alpha"] - 0.34) < 0.08 and abs(answers["beta"] - 0.29) < 0.08
    print(f"  {'exponents recovered':<24} {'ok' if _exp else 'wrong'}  "
          f"(alpha {answers['alpha']:.3f}, beta {answers['beta']:.3f})")
    print(f"  {'tokens/param ~ constant':<24} {'ok' if _stable else 'wrong'}  "
          f"({np.round(_t, 1).tolist()})")
    _ok = _ok and _exp and _stable

print("=" * 56)
if _ok:
    print("PASSED, nice work!")
else:
    print("NOT PASSED yet. Revisit your solution and try again.")

## Solution

Holding `alpha` and `beta` fixed makes the law *linear* in `E`, `A`, and `B`, so
the inner problem is a three-parameter least squares you can solve in closed form.
That is what makes the two-dimensional grid search cheap and robust.

The grid step is 0.025, and the exponents the data were built with, 0.34 and
0.29, are not grid points, so `alpha` and `beta` come back quantised (0.325 and
0.30) and `E`, `A` and `B` absorb the rest: `E` fits to 2.22 where the data were
built on 1.65. Read the five constants as accurate to the grid step, not to the
digits the fit prints.

The allocation result is the interesting part: the optimal tokens-per-parameter
ratio comes out nearly constant across two orders of magnitude of compute, which
is the Chinchilla finding that model size and training tokens should scale
together. Chinchilla itself is 70 billion parameters trained on 1.4 trillion
tokens, about 20 tokens per parameter, whereas the optimum here is near 3,200.

```python
import numpy as np

N = runs[:, 0]
D = runs[:, 1]
L = runs[:, 2]

grid = np.linspace(0.2, 0.8, 25)
best = (np.inf, None)
for a in grid:
    for b in grid:
        X = np.column_stack([np.ones_like(N), N ** (-a), D ** (-b)])
        coef, *_ = np.linalg.lstsq(X, L, rcond=None)
        res = float(np.sum((X @ coef - L) ** 2))
        if res < best[0]:
            best = (res, (a, b, coef))

alpha, beta, coef = best[1]
E, A, B = float(coef[0]), float(coef[1]), float(coef[2])

def loss_of(n, d):
    return E + A / n ** alpha + B / d ** beta

Ngrid = np.logspace(7, 11, 400)
opt_N, opt_D, tpp = [], [], []
for C in budgets:
    d_of = C / (6.0 * Ngrid)
    vals = loss_of(Ngrid, d_of)
    j = int(np.argmin(vals))
    opt_N.append(float(Ngrid[j]))
    opt_D.append(float(d_of[j]))
    tpp.append(float(d_of[j] / Ngrid[j]))

answers = {
    "alpha": float(alpha),
    "beta": float(beta),
    "E": E,
    "opt_N": np.array(opt_N),
    "opt_D": np.array(opt_D),
    "tokens_per_param": np.array(tpp),
}
```